# Synthetic Data Audit (MVP)
This notebook runs a lightweight audit of `dataset/synth_data.csv` using a TF-IDF + logistic regression pipeline and `cleanlab` to detect potential label errors. Each section has a short description followed by the code cell.

In [50]:
# Imports and setup
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
import numpy as np
from cleanlab.filter import find_label_issues

## 1) Load data and quick overview
Load `dataset/synth_data.csv` and display basic counts and label balance.

In [51]:
# Load dataset
df = pd.read_csv('dataset/synth_data.csv')
print('Rows:', len(df))
print('\nLabel distribution:')
print(df['label'].value_counts())
# Ensure columns exist
assert 'sentence' in df.columns and 'label' in df.columns, 'Expected columns: sentence, label'
texts = df['sentence'].astype(str).values
labels = df['label'].values

Rows: 1000

Label distribution:
label
0    505
1    495
Name: count, dtype: int64


## 2) Feature extraction (TF-IDF)
Convert text to TF-IDF features for a simple baseline model.

In [52]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(texts)

## 3) Model training with cross-validated predicted probabilities
Train a logistic regression using 5-fold CV and collect predicted probabilities for `cleanlab`. 

In [53]:
clf = LogisticRegression(max_iter=1000)
pred_probs = cross_val_predict(clf, X, labels, cv=5, method='predict_proba')
print('Predicted probabilities shape:', pred_probs.shape)

Predicted probabilities shape: (1000, 2)


## 4) Detect potential label issues with CleanLab
Use `cleanlab.filter.find_label_issues` to rank examples most likely to be mislabeled.

In [54]:
issue_indices = find_label_issues(labels=labels, pred_probs=pred_probs, return_indices_ranked_by='self_confidence')
print('Found', len(issue_indices), 'potential label issues')
# Preview top 5
for i, idx in enumerate(issue_indices[:5]):
    print(f"\nIssue #{i+1} (Dataset index: {idx})")
    print('Sentence:', df.loc[idx, 'sentence'])
    print('Given label:', df.loc[idx, 'label'])
    print('Predicted probs:', np.round(pred_probs[idx], 3))

Found 8 potential label issues

Issue #1 (Dataset index: 61)
Sentence: A strong script full of wit saves the film.
Given label: 0
Predicted probs: [0.157 0.843]

Issue #2 (Dataset index: 535)
Sentence: Director's vision paints a vivid picture of hope.
Given label: 0
Predicted probs: [0.22 0.78]

Issue #3 (Dataset index: 63)
Sentence: The witty script is the movie's best quality.
Given label: 0
Predicted probs: [0.222 0.778]

Issue #4 (Dataset index: 16)
Sentence: I couldn't get past the weak storyline.
Given label: 1
Predicted probs: [0.755 0.245]

Issue #5 (Dataset index: 78)
Sentence: The pacing is a masterfully woven tapestry.
Given label: 0
Predicted probs: [0.336 0.664]


## 5) Export top suspects for manual review
Save the top 100 ranked examples to a CSV for team validation.

In [55]:
# find_label_issues may return fewer than 100 rows, so we build a full ranking
# using the model's self-confidence for the true label and then place the
# cleanlab-flagged issues first.
label_confidence = pred_probs[np.arange(len(labels)), labels]
ranked_by_suspicion = np.argsort(label_confidence)

top_100 = []
seen = set()
for idx in issue_indices:
    if idx not in seen:
        top_100.append(idx)
        seen.add(idx)
for idx in ranked_by_suspicion:
    if idx not in seen:
        top_100.append(idx)
        seen.add(idx)
    if len(top_100) >= 100:
        break

if len(top_100) < 100:
    raise ValueError(f'Expected at least 100 suspect rows, got {len(top_100)}')

suspicious_df = df.iloc[top_100].copy()
suspicious_df.insert(0, 'Cleanlab_Rank', range(1, len(suspicious_df) + 1))

# Leave these blank for manual review
suspicious_df['Team_Corrected_Label'] = ''
suspicious_df['Error_Type'] = ''
suspicious_df['Notes'] = ''

suspicious_df.to_csv('synth_top_100_suspects.csv', index=False)
print('Exported synth_top_100_suspects.csv with 100 suspect rows')

Exported synth_top_100_suspects.csv with 100 suspect rows
